In [3]:
import sys
sys.path.append("../")
import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import requests
from src.chunking import chunk_text
from src.loaders.txt_loader import load_txt

In [4]:
os.environ["HF_HOME"] = r"G:\huggingface_cache"

In [5]:
# 1. embedding模型（轻量）
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3901.08it/s]


In [6]:
text = load_txt("../data/sample.txt")
print(text[:300])

Retrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Traditional language models rely only on the knowledge stored in their parameters during training. As a result, they may provide outdated information or gen


In [5]:
chunks = chunk_text(
    text,
    chunk_size=200,
    overlap=50
)

docs = []

for i, chunk in enumerate(chunks):

    docs.append({
        "chunk_id": i,
        "text": chunk
    })

print(f"Number of chunks: {len(docs)}")

print(docs[0])

Number of chunks: 19
{'chunk_id': 0, 'text': '\nRetrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Traditional language models rely only on the knowledge '}


In [6]:
for d in docs:
    print(
        f"""
        === Chunk {d['chunk_id']} ===
        {d['text']}
        """
    )


        === Chunk 0 ===
        
Retrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Traditional language models rely only on the knowledge 
        

        === Chunk 1 ===
        tional language models rely only on the knowledge stored in their parameters during training. As a result, they may provide outdated information or generate incorrect answers when asked about private 
        

        === Chunk 2 ===
        nerate incorrect answers when asked about private documents. RAG addresses this limitation by retrieving relevant information from an external knowledge source before generating a response. A typical 
        

        === Chunk 3 ===
        ge source before generating a response. A typical RAG pipeline consists of several stages. First, documents are collected and preprocessed. The text is then divided into smaller chunks to improve retr
        

        === Chunk 4 ===
   

In [7]:
# 3. 向量化
embeddings = model.encode([d["text"] for d in docs])
embeddings = np.array(embeddings).astype("float32")

In [8]:
# 4. 建立FAISS索引
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

In [9]:
def retrieve(query, k=2):
    q_emb = model.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    distances, indices = index.search(q_emb, k)
    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "chunk_id": docs[idx]["chunk_id"],
            "cosine_similarity": float(distances[0][rank]),
            "text": docs[idx]["text"]
        })
    return results

def ask_llm(context, question):
    prompt = f"""
You are a helpful assistant.

Context:
{context}

Question:
{question}
"""

    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:7b",
            "prompt": prompt,
            "stream": False
        }
    )

    return res.json()["response"]

In [11]:
if __name__ == "__main__":
    query = "What is cosine similarity?"

    retrieved = retrieve(query)
    context = "\n".join(r["text"] for r in retrieved)

    answer = ask_llm(context, query)

    print("\n=== Retrieved ===")

    for r in retrieved:
        print(
            f"""
            Chunk {r['chunk_id']}
            cosine_similarity:{r['cosine_similarity']:.3f}
            {r['text']}
            """
        )

    print("\n=== Answer ===")
    print(answer)


=== Retrieved ===

            Chunk 17
            cosine_similarity:0.536
            e.
Vector normalization converts vectors to unit length.
After normalization, inner product becomes equivalent to cosine similarity.
FAISS supports both L2 distance and inner product search.
Top-K ret
            

            Chunk 16
            cosine_similarity:0.484
            g sentence embeddings.
Semantic similarity can be measured using cosine similarity.
Cosine similarity focuses on vector direction rather than magnitude.
Vector normalization converts vectors to unit l
            

=== Answer ===
Cosine similarity is a measure of similarity between two non-zero vectors of an inner product space. It is defined as the cosine of the angle between the two vectors, which gives us a value in the range \([-1, 1]\). The value 1 means the vectors are identical in direction, while -1 indicates that they point in opposite directions. A value of 0 implies that the vectors are orthogonal (perpendicu